In [1]:
import nltk
import re
import string
import pandas as pd
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
from nltk.util import ngrams
from nltk import FreqDist
from collections import Counter

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/2c32c9f8-7173-4889-a499-14458ecbe3dd/nltk_data..
[nltk_data]     .
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/2c32c9f8-7173-4889-a499-14458ecbe3dd/nltk_data..
[nltk_data]     .
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/2c32c9f8-7173-4889-a499-14458ecbe3dd/nltk_data..
[nltk_data]     .
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/2c32c9f8-7173-4889-a499-14458ecbe3dd/nltk_data..
[nltk_data]     .
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
training_df = pd.read_csv('twitter_training.csv')
validation_df = pd.read_csv('twitter_validation.csv')

In [4]:
training_df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [5]:
validation_df.head()

,3364,Facebook,Irrelevant,"I mentioned on Facebook that I was struggling for motivation to go for a run the other day, which has been translated by Tom’s great auntie as ‘Hayley can’t get out of bed’ and told to his grandma, who now thinks I’m a lazy, terrible person 🤣"
0,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
1,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...
2,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,..."
3,4433,Google,Neutral,Now the President is slapping Americans in the...
4,6273,FIFA,Negative,Hi @EAHelp I’ve had Madeleine McCann in my cel...


In [6]:
columns = ['ID','Topic','Label','Text']
training_df.columns = columns
validation_df.columns = columns

In [7]:
training_df.sample(1)

,ID,Topic,Label,Text
3162,1750,CallOfDutyBlackopsColdWar,Positive,November cannot come sooner than I love


In [8]:
validation_df.sample(1)

,ID,Topic,Label,Text
63,8558,NBA2K,Negative,#nba2k is a fucking joke


#### **Lowercasing and Punctuation Removal**

In [9]:
training_df['Text'] = training_df['Text'].str.lower()
validation_df['Text'] = validation_df['Text'].str.lower()


In [10]:
training_df['Text'] = training_df['Text'].apply(lambda x: ''.join([ch for ch in str(x) if ch not in string.punctuation]))
validation_df['Text'] = validation_df['Text'].apply(lambda x: ''.join([ch for ch in str(x) if ch not in string.punctuation]))

In [11]:
training_df['Text'] = training_df['Text'].apply(lambda x: re.sub(r'\d+', '', str(x)))
validation_df['Text'] = validation_df['Text'].apply(lambda x: re.sub(r'\d+', '', str(x)))

In [12]:
training_df.sample(3)

,ID,Topic,Label,Text
65337,7989,MaddenNFL,Neutral,the madden curse game continues
32207,7527,LeagueOfLegends,Neutral,league of legends duoq ranked junglermid twi...
33381,6530,Fortnite,Negative,fuck fortnite u fallguysgame hope up u put fal...


In [13]:
validation_df.sample(3)

,ID,Topic,Label,Text
98,8174,Microsoft,Positive,indigo urgent care looks to microsoft teams an...
995,4359,CS-GO,Irrelevant,this is actually a good move tot bring more vi...
1,8312,Microsoft,Negative,microsoft why do i pay for word when it functi...


In [14]:
emoji_pattern = r'[\U00010000-\U0010ffff]'

training_df['Text'] = training_df['Text'].str.replace(emoji_pattern, '', regex=True)
validation_df['Text'] = validation_df['Text'].str.replace(emoji_pattern, '', regex=True)

#### **Stopwords Removal**

In [15]:
stop_words = stopwords.words('english')

training_df['Text'] = training_df['Text'].apply(lambda x: ' '.join([word for word in str(x).split() if word not in stop_words]))
validation_df['Text'] = validation_df['Text'].apply(lambda x: ' '.join([word for word in str(x).split() if word not in stop_words]))

In [16]:
training_df.sample(3)

,ID,Topic,Label,Text
13602,8734,NBA2K,Irrelevant,kid talent moralde blew away amazing voice ple...
50681,6300,FIFA,Negative,really fucking hate ea fifa worst gameplay time
62271,5074,GrandTheftAuto(GTA),Positive,hella good options loved games gotta go gta pi...


In [17]:
validation_df.sample(3)

,ID,Topic,Label,Text
517,4626,Google,Negative,google sucks google regulates undocumented wor...
111,3206,Facebook,Negative,would fuck way bottom exactly
555,8170,Microsoft,Neutral,microsoft surface headphones review perfect wo...


#### **Stemming**

In [18]:
stemmer = SnowballStemmer("english")

training_df['Text'] = training_df['Text'].apply(lambda x: ' '.join([stemmer.stem(word) for word in str(x).split()]))

validation_df['Text'] = validation_df['Text'].apply(lambda x: ' '.join([stemmer.stem(word) for word in str(x).split()]))

In [31]:
training_df.sample(3)

,ID,Topic,Label,Text
51100,6371,FIFA,Negative,even fifa world taken hour safe say ultim team...
16923,9700,PlayStation5(PS5),Irrelevant,absolut cannot bloodi littl wait game tbh
8061,9384,Overwatch,Negative,overwatch realli listen famous peopl huh


In [32]:
validation_df.sample(3)

,ID,Topic,Label,Text
63,8558,NBA2K,Negative,nbak fuck joke
276,8179,Microsoft,Positive,trust technolog import satyanadella msinspir l...
33,6650,Fortnite,Negative,fortnit run like ass fps drop everywher wtf


#### **N-gram**

In [21]:
tweets = training_df['Text'].dropna().astype(str).tolist()
tweets

['come border kill',
 'im get borderland kill',
 'im come borderland murder',
 'im get borderland murder',
 'im get borderland murder',
 'spent hour make someth fun dont know huge borderland fan maya one favorit charact decid make wallpap pc origin imag versus creation made enjoy pictwittercommlsiwfjg',
 'spent coupl hour someth fun dont know im huge borderland fan maya one favorit charact decid make wallpap pc here origin pictur compar creation made fun pictwittercom mlsiwfjg',
 'spent hour someth fun dont know im huge borderland fan maya one favorit charact',
 'spent hour make someth fun dont know huge rhandlerr fan maya one favorit charact decid make wallpap pc origin imag versus creation made enjoy pictwittercommlsiwfjg',
 'spent hour make someth fun dont know huge rhandlerr fan maya one favorit charact decid make wallpap pc origin imag versus creation made enjoy pictwittercommlsiwfjg',
 '',
 'rockhard la varlop rare power handsom jackpot borderland xbox dlvritrmtrgf',
 'rockhard l

In [22]:
tokenized_tweets = [
    nltk.word_tokenize(tweet)
    for tweet in tweets
    if len(tweet.split()) > 0
]
tokenized_tweets

[['come', 'border', 'kill'],
 ['im', 'get', 'borderland', 'kill'],
 ['im', 'come', 'borderland', 'murder'],
 ['im', 'get', 'borderland', 'murder'],
 ['im', 'get', 'borderland', 'murder'],
 ['spent',
  'hour',
  'make',
  'someth',
  'fun',
  'dont',
  'know',
  'huge',
  'borderland',
  'fan',
  'maya',
  'one',
  'favorit',
  'charact',
  'decid',
  'make',
  'wallpap',
  'pc',
  'origin',
  'imag',
  'versus',
  'creation',
  'made',
  'enjoy',
  'pictwittercommlsiwfjg'],
 ['spent',
  'coupl',
  'hour',
  'someth',
  'fun',
  'dont',
  'know',
  'im',
  'huge',
  'borderland',
  'fan',
  'maya',
  'one',
  'favorit',
  'charact',
  'decid',
  'make',
  'wallpap',
  'pc',
  'here',
  'origin',
  'pictur',
  'compar',
  'creation',
  'made',
  'fun',
  'pictwittercom',
  'mlsiwfjg'],
 ['spent',
  'hour',
  'someth',
  'fun',
  'dont',
  'know',
  'im',
  'huge',
  'borderland',
  'fan',
  'maya',
  'one',
  'favorit',
  'charact'],
 ['spent',
  'hour',
  'make',
  'someth',
  'fun',
  

In [23]:
# Calculate N-grams without crossing tweet boundaries
def calc_ngrams(texts, n):
    ngrams_list = []

    for tokens in texts:
        if len(tokens) >= n:
            ngrams_list.extend(ngrams(tokens, n))

    return ngrams_list

In [24]:
# Calculate Unigrams
def count_unigram(texts):
    tokens = [
        word
        for tweet in texts
        for word in tweet
    ]

    fd = FreqDist(tokens)

    for word, count in fd.most_common(20):
        print(word, count)

    return fd

In [25]:
# Calculate Bigrams / Trigrams
def count_ngram(texts, n):
    ngrams_list = calc_ngrams(texts, n)

    fd = FreqDist(ngrams_list)

    for gram, count in fd.most_common(20):
        print(gram, count)

    return fd

In [26]:
# Unigram frequency
unigram_freq = count_unigram(tokenized_tweets)

game 10905
play 6960
get 5647
like 5215
fuck 4359
love 3867
go 3848
im 3827
one 3718
good 3589
time 3551
realli 3260
new 3210
see 3131
look 2981
make 2922
peopl 2918
johnson 2898
shit 2836
thank 2826


In [27]:
# Bigram frequency
bigram_freq = count_ngram(tokenized_tweets, 2)

('red', 'dead') 1494
('johnson', 'johnson') 1230
('dead', 'redempt') 1199
('i', "'m") 1111
('home', 'depot') 985
('call', 'duti') 876
('assassin', 'creed') 797
('rhandlerr', 'rhandlerr') 780
('itali', 'itali') 769
('gon', 'na') 751
('xbox', 'seri') 705
('seri', 'x') 679
('leagu', 'legend') 676
('cant', 'wait') 588
('ca', "n't") 541
('do', "n't") 518
('access', 'health') 516
('black', 'op') 509
('https', 'tco') 489
("'", 'v') 448


In [28]:
# Trigram frequency
trigram_freq = count_ngram(tokenized_tweets, 3)

('red', 'dead', 'redempt') 1125
('itali', 'itali', 'itali') 761
('xbox', 'seri', 'x') 572
('rhandlerr', 'rhandlerr', 'rhandlerr') 440
('i', "'", 'v') 364
('access', 'health', 'care') 344
('black', 'op', 'cold') 305
('op', 'cold', 'war') 304
('occur', 'see', 'detail') 302
('ban', 'battlefield', 'player') 208
('eu', 'commiss', 'taken') 174
('first', 'time', 'eu') 172
('time', 'eu', 'commiss') 172
('commiss', 'taken', 'step') 172
('time', 'despit', 'fact') 172
('despit', 'fact', 'current') 172
('fact', 'current', 'million') 172
('current', 'million', 'peopl') 172
('million', 'peopl', 'live') 172
('peopl', 'live', 'poverti') 172


In [29]:
# Build N-gram models and count their contexts

def build_ngram_model(tokenized_texts, n):

    ngram_counts = Counter()
    context_counts = Counter()

    for tokens in tokenized_texts:

        # Add start tokens
        padded_tokens = ['<s>'] * (n - 1) + tokens

        for gram in ngrams(padded_tokens, n):

            context = gram[:-1]

            ngram_counts[gram] += 1
            context_counts[context] += 1

    return ngram_counts, context_counts


# Build Bigram Model
bigram_counts, bigram_contexts = build_ngram_model(
    tokenized_tweets, 2
)

# Build Trigram Model
trigram_counts, trigram_contexts = build_ngram_model(
    tokenized_tweets, 3
)


# Calculate Vocabulary Size
vocabulary = set(
    word
    for tweet in tokenized_tweets
    for word in tweet
)

V = len(vocabulary)

print("Vocabulary Size:", V)


# Calculate Probability using Laplace Smoothing
def calculate_probability(sentence, n, ngram_counts, context_counts):

    tokens = nltk.word_tokenize(sentence)

    if not tokens:
        print("No words found!")
        return 0

    padded_tokens = ['<s>'] * (n - 1) + tokens

    alpha = 1  # Laplace Smoothing

    total_probability = 1.0

    print(f"\nCalculating {'Bigram' if n == 2 else 'Trigram'} Probability")
    print("Using Laplace Smoothing")
    print("-" * 60)

    for i in range(n - 1, len(padded_tokens)):

        context = tuple(padded_tokens[i - n + 1:i])
        word = padded_tokens[i]

        gram = context + (word,)

        numerator = ngram_counts[gram]
        denominator = context_counts[context]

        # Laplace Smoothing
        probability = (numerator + alpha) / (denominator + alpha * V)

        print(f"\nN-gram: {gram}")
        print(f"Count of N-gram = {numerator}")
        print(f"Count of Context = {denominator}")
        print(f"Vocabulary Size = {V}")

        print(
            f"Probability = ({numerator} + {alpha}) / "
            f"({denominator} + {alpha} * {V})"
        )

        print(f"Smoothed Probability = {probability}")

        total_probability *= probability

    print("\n" + "-" * 60)
    print("Final Sentence Probability =", total_probability)

    return total_probability


Vocabulary Size: 32153


In [30]:
# Select a sample tweet from the training dataset
sample_text = next(
    tweet for tweet in training_df['Text'].dropna()
    if len(str(tweet).split()) >= 3
)

print("Sample Tweet:")
print(sample_text)


# Calculate Bigram Probability
bigram_probability = calculate_probability(
    sample_text,
    2,
    bigram_counts,
    bigram_contexts
)


# Calculate Trigram Probability
trigram_probability = calculate_probability(
    sample_text,
    3,
    trigram_counts,
    trigram_contexts
)


Sample Tweet:
come border kill

Calculating Bigram Probability
Using Laplace Smoothing
------------------------------------------------------------

N-gram: ('<s>', 'come')
Count of N-gram = 85
Count of Context = 73079
Vocabulary Size = 32153
Probability = (85 + 1) / (73079 + 1 * 32153)
Smoothed Probability = 0.0008172419036034666

N-gram: ('come', 'border')
Count of N-gram = 1
Count of Context = 1710
Vocabulary Size = 32153
Probability = (1 + 1) / (1710 + 1 * 32153)
Smoothed Probability = 5.9061512565336796e-05

N-gram: ('border', 'kill')
Count of N-gram = 1
Count of Context = 61
Vocabulary Size = 32153
Probability = (1 + 1) / (61 + 1 * 32153)
Smoothed Probability = 6.208480784751971e-05

------------------------------------------------------------
Final Sentence Probability = 2.99668112985633e-12

Calculating Trigram Probability
Using Laplace Smoothing
------------------------------------------------------------

N-gram: ('<s>', '<s>', 'come')
Count of N-gram = 85
Count of Context = 